# Modelo Final com LightGBM

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type -> R

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/drive/MyDrive/dmeyf/datasets/competencia_01_fe_Final.csv" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file =  "/content/drive/MyDrive/dmeyf/datasets/competencia_01_fe_Final.csv.gz",
    sep = ","
)

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


## 2.2 Reentreno modelo con nueva información

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Oct 10 03:48:14 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,741789,39.7,1473300,78.7,1473300,78.7
Vcells,1391907,10.7,440741328,3362.6,458802569,3500.4


### 2.2.2 Carga de Librerias

In [ ]:
# instalo paquetes
install.packages( c("R.utils", "rlist" ) )

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’, ‘XML’




In [ ]:
# el LightGBM
install.packages(c("lightgbm"))

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
# Cargo Librerias
library(data.table)
library(R.utils)
library(rlist)
library(lightgbm)

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- "HT4950-ExpFinal"
PARAM$semilla_primigenia <-  366889   # semilla primigenia

In [ ]:
# training y future
PARAM$training_pct <- 70
PARAM$train_final <- c( 202103, 202104, 2021505, 202106)  # Todos los meses salvo julio y agosto
PARAM$future <- c(202108)
PARAM$cortes <- seq(9000, 15000, by= 500)

In [ ]:
# Parametros LightGBM

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt",
  objective= "binary",
  metric= "auc" ,
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,
  feature_fraction= 0.50,
  learning_rate= 0.005,

  num_iterations= 2021,     ## Mejor Hiperparametro a partir optimizador
  num_leaves= 945,          ## Mejor Hiperparametro a partir optimizador
  min_sum_hessian_in_leaf= 0.087496770828561    ## Mejor Hiperparametro a partir optimizador
)


### 2.2.4  Preprocesamiento

In [ ]:
setwd("/content/drive/MyDrive/dmeyf/exp/")
dir.create(PARAM$experimento, showWarnings=FALSE)
setwd( paste0("/content/drive/MyDrive/dmeyf/exp/", PARAM$experimento))

In [ ]:
# lectura del dataset
dataset <- fread("/content/drive/MyDrive/dmeyf/datasets/competencia_01_fe_Final.csv.gz", stringsAsFactors= TRUE)

#### 2.2.4.1  Variables a eliminar

En esa exploracion gráfica del Data Drifting, se vio que estas variables cambian de distribucion entre 202104 y 202106, y se decidió brutalmente eliminarlas

cprestamos_personales

mprestamos_personales

ccajas_depositos se elimina por data faltante en 202105

las variables de seguros se eliminan porque en el feature enginering se sumaron todas en una sola variable.



In [ ]:
campos_buenos <- setdiff( colnames(dataset), c("clase_ternaria") )

campos_buenos <- setdiff( campos_buenos, c("cprestamos_personales", "mprestamos_personales" , "ccajas_depositos " ,
                                            "cseguro_vida" , "cseguro_auto" , "cseguro_vivienda" , "cseguro_accidentes_personales") )

## 2.3  Produccion

### Final Training


In [ ]:
setwd("/content/drive/MyDrive/dmeyf/exp/")
dir.create(PARAM$experimento, showWarnings= FALSE)
setwd( paste0("/content/drive/MyDrive/dmeyf/exp/", PARAM$experimento ))

Entreno con todos los meses salvo 202107 - 202108

In [ ]:
# Redefino las clases
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
# Data set de entrenamiento
dataset_train <- dataset[foto_mes %in% PARAM$train_final]

In [ ]:
# Sin campo de clase
campos_buenos <- setdiff( campos_buenos, c("clase01") )

#### Prediccion de 202108, con 5 semillas calculando un promedio

In [ ]:
# Defino 5 semillas
set.seed(366889) # Semilla base para generar las otras 5
semillas <- sample(1:100000, 5)

# Configuro dataset a predecir
dfuture <- dataset[foto_mes %in% PARAM$future]

dtrain_final <- lgb.Dataset(
  data  = data.matrix(dataset_train[, campos_buenos, with = FALSE]),
  label = dataset_train[, clase01]
)

# Predicciones acumuladas
prediccion_acumulada <- rep(0, nrow(dfuture))

# Mejores Hiperparametros
param_final <- PARAM$lgbm$param_fijos

for (semilla in semillas) {
  param_final$seed <- semilla

  # Entrenar el modelo con la semilla
  modelo_final <- lgb.train(
    data  = dtrain_final,
    param = param_final
  )

  # Predicción de la semilla
  prediccion_actual <- predict(
    modelo_final,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  # Acumulo
  prediccion_acumulada <- prediccion_acumulada + prediccion_actual
}

# Calcular promedio
prediccion_final <- prediccion_acumulada / length(semillas)

#  Tabla final
tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]
tb_prediccion[, prob := prediccion_final]

setorder(tb_prediccion, -prob)
tb_prediccion[, pos := sequence(.N)]

In [ ]:
tb_prediccion[pos <= 12500]

numero_de_cliente,foto_mes,clase_ternaria,prob,pos
<int>,<int>,<fct>,<dbl>,<int>
31959546,202108,,0.8703883,1
42425357,202108,,0.8688431,2
32786043,202108,,0.8621629,3
45696840,202108,,0.8616011,4
52878216,202108,,0.8407329,5
60622229,202108,,0.8360478,6
64789926,202108,,0.8283301,7
38799788,202108,,0.8252145,8
42738466,202108,,0.7990005,9


In [ ]:
library(readr)
write_csv(tb_prediccion[pos <= 11500, "numero_de_cliente"], "exp_final_115.csv")  # Exporto archivo de clientes hasta 11500